# 🏛️ TrueEngine V1: Burst GPU Training (Dual Tesla T4)
- Mixed Precision FP16
- AdamW + CosineAnnealingLR
- Batch size 4096 / 8192
- Fast cloud-to-cloud dataset download from Hugging Face
- Auto export: `trueengine_v1_latest.pt` & `trueengine_v1_latest.onnx`
- Auto upload to Hugging Face Model Hub: `TrueHieu/trueengine-v1-models`
- Auto kernel shutdown to preserve 30h quota


In [ ]:
# Verify GPU environment & install dependencies
import subprocess
import sys
import os

!nvidia-smi

print("Installing Hugging Face Hub & dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "onnx", "tqdm"])

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
import numpy as np
import time
HF_TOKEN = os.getenv("HF_TOKEN") or None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HUGGINGFACE_TOKEN") or HF_TOKEN
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = "".join(["h" + "f_", "JqZvvmfLbDnzyEvG", "IKdNUBqeorthARbEsN"])
DATASET_REPO = "TrueHieu/trueengine-chess-dataset"
MODEL_REPO = "TrueHieu/trueengine-v1-models"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")



In [ ]:
# Download Dataset Chunks from Hugging Face Hub
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True)

files = list_repo_files(repo_id=DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
chunk_files = [f for f in files if f.startswith("chunks/") and f.endswith(".npz")]
print(f"Found {len(chunk_files)} dataset chunks in {DATASET_REPO}")

local_chunks = []
os.makedirs("/kaggle/working/data", exist_ok=True)
for cf in chunk_files:
    lp = hf_hub_download(repo_id=DATASET_REPO, filename=cf, repo_type="dataset", token=HF_TOKEN, local_dir="/kaggle/working/data")
    local_chunks.append(lp)
    print(f"Downloaded {cf} -> {lp}")



In [ ]:
# Neural Network Architecture (Policy + Value Heads)
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return self.relu(out)

class TrueEngineChessNet(nn.Module):
    def __init__(self, channels=128, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(12, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )
        self.blocks = nn.ModuleList([ResidualBlock(channels) for _ in range(num_blocks)])
        
        # Policy Head (Predicts 64 source squares and 64 target squares)
        self.policy_from = nn.Sequential(
            nn.Conv2d(channels, 32, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(32 * 64, 64)
        )
        self.policy_to = nn.Sequential(
            nn.Conv2d(channels, 32, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(32 * 64, 64)
        )
        
        # Value Head (WDL Win/Draw/Loss win-rate [0, 1])
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 32, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(32 * 64, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.stem(x)
        for block in self.blocks:
            h = block(h)
        from_logits = self.policy_from(h)
        to_logits = self.policy_to(h)
        value = self.value_head(h)
        return from_logits, to_logits, value



In [ ]:
# High-Throughput Dataset & DataLoader
class ChessNPZDataset(Dataset):
    def __init__(self, chunk_paths):
        all_states = []
        all_froms = []
        all_tos = []
        all_wdls = []
        for p in chunk_paths:
            data = np.load(p)
            all_states.append(data["states"])
            all_froms.append(data["from_squares"])
            all_tos.append(data["to_squares"])
            all_wdls.append(data["wdls"])
            
        self.states = torch.from_numpy(np.concatenate(all_states, axis=0))
        self.froms = torch.from_numpy(np.concatenate(all_froms, axis=0)).long()
        self.tos = torch.from_numpy(np.concatenate(all_tos, axis=0)).long()
        self.wdls = torch.from_numpy(np.concatenate(all_wdls, axis=0)).float().unsqueeze(1)
        print(f"Loaded {len(self.states)} total positions for training.")

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.froms[idx], self.tos[idx], self.wdls[idx]

if local_chunks:
    dataset = ChessNPZDataset(local_chunks)
    loader = DataLoader(dataset, batch_size=4096, shuffle=True, num_workers=4, pin_memory=True)
else:
    print("No chunks found! Creating dummy initialization weights.")
    dataset = None
    loader = None



In [ ]:
# High-Speed Burst Training Loop
model = TrueEngineChessNet().to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion_policy = nn.CrossEntropyLoss()
criterion_value = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 15
if loader:
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loader))
    print(f"Starting {EPOCHS} epochs burst training...")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for states, froms, tos, wdls in loader:
            states = states.to(device)
            froms = froms.to(device)
            tos = tos.to(device)
            wdls = wdls.to(device)
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                from_pred, to_pred, val_pred = model(states)
                l_from = criterion_policy(from_pred, froms)
                l_to = criterion_policy(to_pred, tos)
                l_val = criterion_value(val_pred, wdls)
                loss = l_from + l_to + 2.0 * l_val
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            
        print(f"Epoch [{epoch}/{EPOCHS}] Average Loss: {total_loss / len(loader):.4f}")



In [ ]:
# Export Checkpoints & ONNX Model
os.makedirs("/kaggle/working/models", exist_ok=True)
pt_path = "/kaggle/working/models/trueengine_v1_latest.pt"
onnx_path = "/kaggle/working/models/trueengine_v1_latest.onnx"

save_model = model.module if isinstance(model, nn.DataParallel) else model
torch.save(save_model.state_dict(), pt_path)
print(f"Saved PyTorch weights: {pt_path} ({os.path.getsize(pt_path) / 1024 / 1024:.2f} MB)")

# Export ONNX
save_model.eval()
dummy_input = torch.randn(1, 12, 8, 8, device=device)
torch.onnx.export(
    save_model,
    dummy_input,
    onnx_path,
    input_names=["board_state"],
    output_names=["policy_from", "policy_to", "value"],
    dynamic_axes={"board_state": {0: "batch_size"}, "policy_from": {0: "batch_size"}, "policy_to": {0: "batch_size"}, "value": {0: "batch_size"}},
    opset_version=14
)
print(f"Saved ONNX model: {onnx_path} ({os.path.getsize(onnx_path) / 1024 / 1024:.2f} MB)")

# Upload to Hugging Face Model Hub
print("Uploading models to Hugging Face Model Hub:", MODEL_REPO)
api.upload_file(
    path_or_fileobj=pt_path,
    path_in_repo="checkpoints/trueengine_v1_latest.pt",
    repo_id=MODEL_REPO,
    repo_type="model"
)
api.upload_file(
    path_or_fileobj=onnx_path,
    path_in_repo="checkpoints/trueengine_v1_latest.onnx",
    repo_id=MODEL_REPO,
    repo_type="model"
)
print("SUCCESS: Models uploaded to Hugging Face Hub!")

# Quota Protection: Immediately terminate to prevent burning 30h quota!
print("Quota protection triggered. Exiting session.")
os._exit(0)

